# Notebook 03: Feature Engineering & Baseline Forecasting
## Time-Series Feature Store & Benchmark Models

This notebook covers:
1. **Auto-Regressive Lag Features**: `Lag_1`, `Lag_7`, `Lag_14`, `Lag_28`
2. **Causal Rolling Statistics**: `Rolling_Mean_7`, `Rolling_Mean_14`, `Rolling_Std_7` computed on shifted series with **ZERO lookahead leakage**
3. **Temporal & Cyclical Encodings**: `Day_of_Week`, `Month`, `Is_Weekend`, `sin/cos` periodic encodings
4. **Strict Chronological Train / Val / Test Partitioning** (70% / 15% / 15%)
5. **Baseline Benchmark Evaluation**: Naive Persistence, Seasonal Naive, Simple Moving Averages, Historical Mean

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import custom feature pipeline & baselines
from ml.features.feature_pipeline import FeaturePipeline
from ml.models.baselines import NaivePersistenceForecast, SeasonalNaiveForecast, MovingAverageForecast, HistoricalMeanForecast
from ml.evaluation.metrics import evaluate_forecast

sns.set_theme(style='whitegrid')

In [ ]:
# Load cleaned data and apply feature pipeline
df = pd.read_csv('../data/processed/cleaned_sales_data.csv')
pipeline = FeaturePipeline()
df_feat = pipeline.transform(df, drop_na=True)

print(f'Engineered features shape: {df_feat.shape}')
df_feat[['Date', 'Store_ID', 'Product_ID', 'Units_Sold', 'Lag_1', 'Lag_7', 'Rolling_Mean_7', 'Rolling_Std_7']].head(10)

In [ ]:
# Chronological Splitting
train_df, val_df, test_df = FeaturePipeline.chronological_split(df_feat, train_ratio=0.70, val_ratio=0.15)

print(f'Train: {len(train_df):,} rows ({train_df["Date"].min()} to {train_df["Date"].max()})')
print(f'Val  : {len(val_df):,} rows ({val_df["Date"].min()} to {val_df["Date"].max()})')
print(f'Test : {len(test_df):,} rows ({test_df["Date"].min()} to {test_df["Date"].max()})')

In [ ]:
# Evaluate Baselines
models = [
    NaivePersistenceForecast(),
    SeasonalNaiveForecast(),
    MovingAverageForecast(window=7),
    MovingAverageForecast(window=14)
]

y_test = test_df['Units_Sold'].values
results = []

for m in models:
    preds = m.predict(test_df)
    metrics = evaluate_forecast(y_test, preds)
    results.append({'Model': m.name, **metrics})

pd.DataFrame(results)